In [4]:
import os
import pandas as pd
from itertools import product
from collections import Counter
from joblib import Parallel, delayed
from tqdm import tqdm

In [3]:
df = pd.read_csv("../raw_data/protein_sequences/protein_sequences.csv")
df.head()

,uniprot_id,protein_name,protein_sequence
0,VSPPA_TRIST,Snake_Venom_Serine_Proteases(SVSP),MELIRVLANLLILQLSYAQKSSELVFGGDECNINEHRSLVVLFNSN...
1,VSPBF_MACLB,Snake_Venom_Serine_Proteases(SVSP),MVLIRVLANLLLLQLSHAQKSSELVVGGDECNINEHRSLVFLYNSS...
2,VSP3_BOTJA,Snake_Venom_Serine_Proteases(SVSP),MVLIRVIANLLILQLSNAQKSSELVIGGDECNITEHRFLVEIFNSS...
3,VSPS2_TRIST,Snake_Venom_Serine_Proteases(SVSP),MELIRVLANLLILQLSYAQKSSELVVGGDECNINEHRSLVAIFNST...
4,VSPA_MACLB,Snake_Venom_Serine_Proteases(SVSP),MVLIRVLANLVMLHLSYGEKSSELVIGGRPCNINQHRSLALLYNSS...


# Protein Feature Engineering Pipeline
 ## **1.Feature Generation**

    Class: ProteinFeatureGenerator

    Extracts frequency-based features from protein sequences using:
        1-mers (single amino acids)
        2-mers (dipeptides)
        3-mers (tripeptides)

    Cleans sequences and generates feature vectors.
    Saves processed feature dataset to CSV.

In [5]:
#class to generate k-mer based features from protein sequences

class ProteinFeatureGenerator:
    def __init__(self,
                 raw_path: str,
                 processed_path: str,
                 approaches: list = ["single", "di", "tri"]):
        self.raw_path = raw_path
        self.processed_path = processed_path
        self.approaches = approaches

        # Standard 20 amino acids
        self.amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

        # Predefined k-mer combinations for 1-mer, 2-mer, and 3-mer
        self.kmer_sets = {
            "single": self.amino_acids,
            "di": [''.join(p) for p in product(self.amino_acids, repeat=2)],
            "tri": [''.join(p) for p in product(self.amino_acids, repeat=3)],
        }

        self.dataset = pd.read_csv(self.raw_path)

    # Function to remove any unknown or non-standard amino acids from a sequence
    def clean_sequence(self, seq):
        return ''.join([aa for aa in seq if aa in self.amino_acids])

    # Function to compute k-mer frequencies for a given sequence
    def compute_kmer_freq(self, seq, kmers):
        k = len(kmers[0])
        seq = self.clean_sequence(seq)
        counter = Counter([seq[i:i+k] for i in range(len(seq)-k+1)])
        total = sum(counter[k] for k in kmers)
        return [counter.get(k, 0)/total if total > 0 else 0 for k in kmers]

    # Apply k-mer frequency computation in parallel for all sequences
    def compute_features(self, sequences, kmers, desc):
        return Parallel(n_jobs=-1)(
            delayed(self.compute_kmer_freq)(seq, kmers) for seq in tqdm(sequences, desc=desc)
        )

    # Main function to generate and save feature vectors
    def generate_features(self):
        feature_frames = [self.dataset[["uniprot_id", "protein_name"]]]

        for kmer_type in self.approaches:
            if kmer_type not in self.kmer_sets:
                print(f"Warning: '{kmer_type}' is not a supported k-mer type. Skipping.")
                continue

            print(f"Generating {kmer_type}-mer frequency features...")
            kmers = self.kmer_sets[kmer_type]
            freq_matrix = self.compute_features(self.dataset["protein_sequence"], kmers, desc=f"{kmer_type.capitalize()}")
            freq_df = pd.DataFrame(freq_matrix, columns=[f"freq_{k}" for k in kmers])
            feature_frames.append(freq_df)

        full_df = pd.concat(feature_frames, axis=1)

        os.makedirs(os.path.dirname(self.processed_path), exist_ok=True)
        full_df.to_csv(self.processed_path, index=False)

        print(f"Features saved to: {self.processed_path}")


if __name__ == "__main__":
    feature_gen = ProteinFeatureGenerator(
        raw_path="../raw_data/protein_sequences/protein_sequences.csv",
        processed_path="../processed_data/protein_features/processed_protein_features_wout_deduplication.csv",
        approaches=["single", "di", "tri"]
    )

    feature_gen.generate_features()

Generating single-mer frequency features...


Single: 100%|██████████████████████████████████████████████████████████████████████| 6355/6355 [02:05<00:00, 50.47it/s]


Generating di-mer frequency features...


Di: 100%|██████████████████████████████████████████████████████████████████████████| 6355/6355 [01:04<00:00, 99.09it/s]


Generating tri-mer frequency features...


Tri: 100%|█████████████████████████████████████████████████████████████████████████| 6355/6355 [01:34<00:00, 67.58it/s]


Features saved to: ../processed_data/protein_features/processed_protein_features_wout_deduplication.csv


In [7]:
    #  Split based on fold column in metadata.csv 
    print("Splitting data based on folds...")

    # Load metadata and processed features
    metadata_df = pd.read_csv("../raw_data/metadata/metadata.csv")
    features_df = pd.read_csv("../processed_data/protein_features/processed_protein_features_wout_deduplication.csv")

    # Merge on 'uniprot_id'
    merged_df = metadata_df[["uniprot_id", "fold"]].merge(features_df, on="uniprot_id", how="inner")

    # Split into train, test, and val sets
    for split in ["train", "test", "val"]:
        split_df = merged_df[merged_df["fold"] == split]
        output_path = f"../processed_data/Split data/{split}.csv"
        split_df.to_csv(output_path, index=False)
        print(f"Saved {split} set to: {output_path}")

Splitting data based on folds...
Saved train set to: ../processed_data/Split data/train.csv
Saved test set to: ../processed_data/Split data/test.csv
Saved val set to: ../processed_data/Split data/val.csv
